In [8]:
import os
os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "1"

from deepeval import evaluate
from deepeval.test_case import LLMTestCase, SingleTurnParams, Turn, MultiTurnParams, ConversationalTestCase
from deepeval.metrics import (
    GEval, ConversationalGEval, DAGMetric, AnswerRelevancyMetric, 
    FaithfulnessMetric, ContextualPrecisionMetric,
    ContextualRecallMetric, ContextualRelevancyMetric
)
from deepeval.metrics.g_eval import Rubric
from deepeval.metrics.dag import BinaryJudgementNode, DeepAcyclicGraph, TaskNode, NonBinaryJudgementNode

V tomto textu se nalézají mé poznámky spojené s pythoním balíčkem deepeval. Ten slouží na vyhodnocování výstupů jazykových nodelů a s nimi spojených aplikací (např. RAGů či agentních systémů).

Jelikož deepeval jede primárně v modu "LLM-as-a-judge", potřebuje mít k dispozici API klíč. Ten musí být k nalezení v proměnných prostředí jako OPENAI_API_KEY. Pakliže máme tuto proměnnou uloženouv ".env" souboru, nemusíme se nia zatěžovat dotenv příkazy - deepeval si obsah souboru [načte automaticky](https://deepeval.com/docs/environment-variables).  
Je třeba vědět, že deepeval volá domů - měl by tedy posílat jen věci zmíněné [na této stránce](https://deepeval.com/docs/data-privacy). Jak takovou věc vypneme? Musíme použít proměnnou prostředí *DEEPEVAL_TELEMETRY_OPT_OUT*.

In [2]:
import os
os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "1"

## Základy a G-Eval

Pro tu nejjednodušší evaluaci potřebujeme tři věci. Metriku, podle které se bude výstup LLMka posuzovat; testovací case, ve kterém bude uvedeno, jaký je vstup, reálný a očekávaný výstup; a nakonec samotný akt evaluace.  
Jako metriku v následujícím příkladu použijeme *GEval*. Jendá se o nejobecnější metriku v deepevalu, která se tak hodí na subjektivní ohodnocování. S jakými parametry provoláváme její konstruktor?
- model - tento parametr je nepovinný. Kdybychom ho pominuli, použil by se (v době psaní těchto řádků) gpt-5.4 (viz [zde](https://deepeval.com/docs/metrics-llm-evals)). Jelikož nám v tuhle chvíli nejde o kdoví jakou přenost, používáme mnohem levnější OpenAI model, který do konstruktoru vkládáme jako string. Pokud bychom chtěli použít model od jiného poskytovatele, tak by to šlo taky, ale přeci jen trochu [složitěji](https://deepeval.com/integrations/models/azure-openai).
- name - jedná se o jméno metriky a jeden z povinných parametrů. Nicméně slouží jen abychom ve výstupu věděli, k jaké metrice se vlastně to či ono ohodnocení váže.
- criteria - dalčí povinný parametr a tentokrát opravdu důlůežitý pro fungování metriky - říká, co přesně má metrika kontrolovat.  
- evaluation_params - též povinný parametr; jedná se o list, který obsahuje prvky z enumu SingleTurnParams. Tím je stanoveno, jaké parametry (krom inputu) budou pro evaluaci důležité. V oučasnosti se ve zmíněném enumu nacházejí tyto hodnoty: 'ACTUAL_OUTPUT', 'CONTEXT', 'EXPECTED_OUTPUT', 'EXPECTED_TOOLS', 'INPUT', 'MCP_PROMPTS_CALLED', 'MCP_RESOURCES_CALLED', 'MCP_SERVERS', 'MCP_TOOLS_CALLED', 'METADATA', 'RETRIEVAL_CONTEXT', 'TAGS', 'TOOLS_CALLED'.  
- threshold - jedná se o nepovinný parametr s defaultní hodnotou 0.5 (tj. vlastně bychom ho v příkladu explicitně nemuseli uvádět). Jedná se o práh, pod kterým budou testy vyhodnoceny jako selhávající. Přitom všechny metriky by měly vracet skore evaluace mezi 0 a 1.

Do instance LLMTestCase vložíme vstup a reálný a očekávaný výstup (viz evaluation_params výše). V reálném provozu by pochopitelně v actual_output nebyl natvrdo string, ale výstup LLMka.

Nakonec funkce *evaluate* spustí samotné vyhodnocování. Vložíme do ní jednak list testovacích casů, jednak list metrik.

V příkaldu vidíme i formát výstupu. Všimněme si, že je uveden čas vyhodnocování a i spálené peníze. Výstup celého procesu je objekt typu *EvaluationResult*, který bychom v případě nutnosti mohli použít na strojové zpracování výsledku.

In [8]:
from deepeval import evaluate
from deepeval.test_case import LLMTestCase, SingleTurnParams
from deepeval.metrics import GEval

correctness_metric = GEval(
    model="gpt-4.1-mini",
    name="Correctness",
    criteria="Determine if the 'actual output' is correct based on the 'expected output'.",
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.EXPECTED_OUTPUT],
    threshold=0.5
)

test_case = LLMTestCase(
    input="Když jsou králící šťastní, skáčou radostí?",
    actual_output="Ano, když jsou králící šťastní, tak skáčou.",
    expected_output="Ano, králíci skáčou radostí a tento projev je jasným znakem maximálního štěstí, bezpečí a životní energie. Tomuto specifickému chování se v chovatelské komunitě mezinárodně říká binky."
)

evaluate([test_case], [correctness_metric])

✨ You're running DeepEval's latest Correctness [GEval] Metric! (using gpt-4.1-mini, strict=False, 
async_mode=True)...

c:\primary\programovani\workshops\environment\Lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets"
for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_0                                                                                                 │
│  ├──   Input:              Když jsou králící šťastní, skáčou radostí?                                           │
│  │     Actual Output:      Ano, když jsou králící šťastní, tak skáčou.                                          │
│  │     Expected Output:    Ano, králíci skáčou radostí a tento projev je jasným znakem maximálního štěstí,      │
│  │                         bezpečí a životní energie. Tomuto specifickému chování se v chovatelské komunitě     │
│  │                         mezinárodně říká binky.                                                              │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric              ┃ Score ┃ Threshold ┃ Reason                                                 │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        FAIL  │ Correctness [GEval] │ 0.02  │ 0.50      │ The actual output does not match the expected output   │
│              │                     │       │           │ in content, detail, or length. It lacks the specific   │
│              │                     │       │           │ explanation about 'binky' behavior and the             │
│              │                     │       │           │ associated meanings of happiness, safety, and          │
│              │                     │       │           │ vitality, failing to meet the detailed requirements    │
│              │                     │       │           │ of the expected output.                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                         ┃ Average Score         ┃ Pass Rate                                 ┃ Total     │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━ │
│  Correctness [GEval]            │ 0.02                  │ 0.00% | passed=0 | failed=1               │ 1         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=5467295;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 12.37s | token cost: 0.0004332 USD)
» Test Results (1 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_0', success=False, metrics_data=[MetricData(name='Correctness [GEval]', threshold=0.5, success=False, score=0.017028275114404825, reason="The actual output does not match the expected output in content, detail, or length. It lacks the specific explanation about 'binky' behavior and the associated meanings of happiness, safety, and vitality, failing to meet the detailed requirements of the expected output.", strict_mode=False, flaky=False, evaluation_model='gpt-4.1-mini', error=None, evaluation_cost=0.0004332, input_tokens=531, output_tokens=138, verbose_logs='Criteria:\nDetermine if the \'actual output\' is correct based on the \'expected output\'. \n \nEvaluation Steps:\n[\n    "Compare the actual output to the expected output exactly, checking for identical content.",\n    "Verify that the actual output meets all specified requirements and constraints outlined in the expected output.",\n    "Check for any discrepancies or devi

Do *GEval* lze namísto *criteria* vložit *evaluation_steps* s listem instrukcí, které má evaluátor brát při vyhodnocování v úvahu. Tyhle instrukce se objeví i v defaultním běhu, tehdy jsou ale vygenerovány právě z *criteria*.

In [12]:
correctness_metric = GEval(
    model="gpt-4.1-mini",
    name="Correctness",
    evaluation_steps=[
        "Check whether info in 'actual_output' is present in 'expected_output'",
        "If 'expected_output' contains more information than 'actual_output' shouldn't play the role in evaluation."
    ],
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.EXPECTED_OUTPUT],
    threshold=0.5
)

test_case = LLMTestCase(
    input="Když jsou králící šťastní, skáčou radostí?",
    actual_output="Ano, když jsou králící šťastní, tak skáčou.",
    expected_output="Ano, králíci skáčou radostí a tento projev je jasným znakem maximálního štěstí, bezpečí a životní energie. Tomuto specifickému chování se v chovatelské komunitě mezinárodně říká binky."
)

eval_object = evaluate([test_case], [correctness_metric])

✨ You're running DeepEval's latest Correctness [GEval] Metric! (using gpt-4.1-mini, strict=False, 
async_mode=True)...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                        ┃ Average Score        ┃ Pass Rate                                   ┃ Total     │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━ │
│  Correctness [GEval]           │ 0.62                 │ 100.00% | passed=1 | failed=0               │ 1         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=5467297;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 4.22s | token cost: 0.00028240000000000003 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

Pokud chceme mít nad procesem vyhodnocování větší kontrolu, použijeme parametr *rubric*. Do něj půjde list objektů Rubric, které se skládají z (nepřekrývajících se) intervalů a popisů, co by se v intervalu mělo objevit. Rozsahy se musí nalézat mezi 0 a 10 - na konci totiž bude takto určené skore vyděleno deseti - viz [zde](https://deepeval.com/docs/metrics-llm-evals#how-is-it-calculated). 

In [18]:
from deepeval.metrics.g_eval import Rubric

correctness_metric = GEval(
    model="gpt-4.1-mini",
    name="Correctness",
    criteria="Determine if the 'actual output' is correct based on the 'expected output'.",
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.EXPECTED_OUTPUT],
    threshold=0.5,
    rubric=[
        Rubric(score_range=(0,3), expected_outcome="Opposite meaning."),
        Rubric(score_range=(4,6), expected_outcome="Mostly correct even if lacking some detail."),
        Rubric(score_range=(7,9), expected_outcome="Correct but missing minor details."),
        Rubric(score_range=(10,10), expected_outcome="100% correct."),
    ]
)

test_case = LLMTestCase(
    input="Když jsou králící šťastní, skáčou radostí?",
    actual_output="Ano, když jsou králící šťastní, tak skáčou.",
    expected_output="Ano, králíci skáčou radostí a tento projev je jasným znakem maximálního štěstí, bezpečí a životní energie. Tomuto specifickému chování se v chovatelské komunitě mezinárodně říká binky."
)

eval_object = evaluate([test_case], [correctness_metric])

✨ You're running DeepEval's latest Correctness [GEval] Metric! (using gpt-4.1-mini, strict=False, 
async_mode=True)...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_0                                                                                                 │
│  ├──   Input:              Když jsou králící šťastní, skáčou radostí?                                           │
│  │     Actual Output:      Ano, když jsou králící šťastní, tak skáčou.                                          │
│  │     Expected Output:    Ano, králíci skáčou radostí a tento projev je jasným znakem maximálního štěstí,      │
│  │                         bezpečí a životní energie. Tomuto specifickému chování se v chovatelské komunitě     │
│  │                         mezinárodně říká binky.                                                              │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric              ┃ Score ┃ Threshold ┃ Reason                                                 │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        FAIL  │ Correctness [GEval] │ 0.40  │ 0.50      │ The actual output conveys the basic idea that happy    │
│              │                     │       │           │ rabbits jump, which aligns with the expected           │
│              │                     │       │           │ output's core message. However, it lacks significant   │
│              │                     │       │           │ detail about the behavior being a clear sign of        │
│              │                     │       │           │ maximum happiness, safety, and vitality, and it        │
│              │                     │       │           │ omits the specific term 'binky' used internationally   │
│              │                     │       │           │ in the rabbit-keeping community. This results in a     │
│              │                     │       │           │ mostly correct but incomplete response.                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                         ┃ Average Score         ┃ Pass Rate                                 ┃ Total     │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━ │
│  Correctness [GEval]            │ 0.40                  │ 0.00% | passed=0 | failed=1               │ 1         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=5467305;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 10.3s | token cost: 0.0004792 USD)
» Test Results (1 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

## Conversational G-Eval
G-Eval jako taková kontrolovala jen jeden konrkétní výstup, který závisel na jednom konrkétním vstupu. Co když ale potřebujeme vyhodnotit celou konverzaci? V takovém případě použijeme *ConversationalGEval*. Jednotlivá kola konverzace jsou reprezentována objekty typu *Turn*, které obsahují pole *role* a *content*. Do *evaluation_params* konstruktoru *ConversationalGEval* musíme vložit minimálně *MultiTurnParams.CONTENT*, aby systém věděl, že má posuzovat právě pole *content*.

In [9]:
from deepeval.test_case import Turn, MultiTurnParams, ConversationalTestCase
from deepeval.metrics import ConversationalGEval
from deepeval import evaluate

test_case = ConversationalTestCase(
    turns=[
        Turn(role="user", content="Co to je skořice?"), 
        Turn(role="assistant", content="Skořice je koření... Mám chuť na mrkev - nemáš nějakou? Prosíííím..."),
        Turn(role="user", content="A co to konrkétně je - nějaký druh ořechu, či snad hlíza jako u brambor?"), 
        Turn(role="assistant", content="Ne, jedná se o kůru stromu."),
    
    ]
)
metric = ConversationalGEval(
    model="gpt-4.1-mini",
    name="Professionalism",
    criteria="Determine whether the assistant has acted professionally and wasn't thinking on irrelevant topics as a food.",
    evaluation_params=[MultiTurnParams.CONTENT],
    threshold=0.9
)

eval_object = evaluate(test_cases=[test_case], metrics=[metric])

✨ You're running DeepEval's latest Professionalism [Conversational GEval] Metric! (using gpt-4.1-mini, 
strict=False, async_mode=True)...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ conversational_test_case_0                                                                                  │
│  ├── Conversation Turns                                                                                         │
│  │   ├── User: Co to je skořice?                                                                                │
│  │   ├── Assistant: Skořice je koření... Mám chuť na mrkev - nemáš nějakou? Prosíííím...                        │
│  │   ├── User: A co to konrkétně je - nějaký druh ořechu, či snad hlíza jako u brambor?                         │
│  │   └── Assistant: Ne, jedná se o kůru stromu.                                                                 │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric                               ┃ Score ┃ Threshold ┃ Reason                                │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        FAIL  │ Professionalism [Conversational      │ 0.55  │ 0.90      │ The assistant maintains a             │
│              │ GEval]                               │       │           │ professional tone and provides        │
│              │                                      │       │           │ relevant information about            │
│              │                                      │       │           │ cinnamon, but introduces an           │
│              │                                      │       │           │ off-topic and informal comment        │
│              │                                      │       │           │ about craving carrots, which          │
│              │                                      │       │           │ detracts from professionalism and     │
│              │                                      │       │           │ relevance, violating evaluation       │
│              │                                      │       │           │ steps 1 and 3.                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                                         ┃ Average Score    ┃ Pass Rate                        ┃ Total   │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━ │
│  Professionalism [Conversational GEval]         │ 0.55             │ 0.00% | passed=0 | failed=1      │ 1       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=5422792;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 4.89s | token cost: 0.0005804 USD)
» Test Results (1 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

## DAG
Zkratka DAG zde znamená "deep acyclical graph" (resp. "direct acyclical graph" - dokumentace není v tomto úplně konzistentní). Jedná se o metriku, která stejně jako G-Eval jede v modu LLM-as-a-judge, nicméně poskytuje uživateli větší kontrolu nad evaluací.  
Nejprve si musíme vytvořit root grafu - v následující ukázce to bude instance *BinaryJudgementNode*. *BinaryJudgementNode* vždy vyhodnotí jedno kritérium a vrátí buďto True, anebo False. Do konstruktoru nodu nikdy nevkládáme hrany grafu - ty vložíme až následně srkz metodu *add_verdict*. U našeho triviálního DAGu půjde jen o to, že při hodnotě verdiktu True dostane testovací case hodnocení 10 a při hodnotě False 0 (na konci se pak oěpt uplatní dělení deseti). Do *DAGMetric* objektu musíme vložit skrz parametr *dag* instanci *DeepAcyclicGraph*, do jejíhož parametru *root_nodes* vložíme právě naši *BinaryJudgementNode*.  
BTW do metriky jsem přidal i parametr *verbose_mode* s hodnotou True, abychom viděli úvahy modelu i v případě úspěšného průběhu testu.

In [13]:
from deepeval.metrics import DAGMetric
from deepeval.metrics.dag import BinaryJudgementNode, DeepAcyclicGraph

buniness = BinaryJudgementNode(
    criteria="Is the text talking about bunnies?",
    evaluation_params=[
        SingleTurnParams.INPUT,
    ],
)
buniness.add_verdict(verdict=True, score=10)
buniness.add_verdict(verdict=False, score=0)

dag_metric = DAGMetric(
    model="gpt-4.1-mini",
    name="custom_dag_metric",
    dag=DeepAcyclicGraph(root_nodes=[buniness]),
    verbose_mode=True
)

test_case = LLMTestCase(
    input="Do rabbits perform popcorning?",
    actual_output="Yes, they do.",
)

eval_object = evaluate(test_cases=[test_case], metrics=[dag_metric])

✨ You're running DeepEval's latest custom_dag_metric [DAG] Metric! (using gpt-4.1-mini, strict=False, 
async_mode=True)...

**************************************************

custom_dag_metric [DAG] Verbose Logs

**************************************************

__________________________________
| BinaryJudgementNode | Level == 0 |
************************************************
Label: None

Criteria:
Is the text talking about bunnies?

Verdict: True
Reason: The text explicitly mentions 'rabbits,' which are commonly referred to as bunnies, and discusses their 
behavior called 'popcorning.' Therefore, the text is talking about bunnies.
 
 
________________________
| VerdictNode | Level == 1 |
**********************************
Verdict: True
Type: Deterministic
 
Score: 1.0
Reason: The score is 1.0 because the DAG traversal shows that the BinaryJudgementNode at level 0 determined the 
text is talking about bunnies, explicitly mentioning 'rabbits' and their behavior 'popcorning,' leading to a True 
verdict at the VerdictNode, confirming the metric's condition is fully met.

======================================================================

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                            ┃ Average Score       ┃ Pass Rate                                 ┃ Total    │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━ │
│  custom_dag_metric [DAG]           │ 1.00                │ 100.00% | passed=1 | failed=0             │ 1        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=5422796;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 3.4s | token cost: 0.00040320000000000004 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

Existují i další dva typy nodů.  
*TaskNode* provede to, co od něj chceme,přičemž žádné skore negeneruje. Krom parametru *instruction* musí obsahovat i parametr *output_label* říkající, pod jakým názvem následné nody najdou jeho výstup.  
*NonBinaryJudgementNode* funguje podobně jako *BinaryJudgementNode*, akorát může mít více než True/False výstupy. Tyto výstupy nedefinujeme v kosntruktoru, nýbrž v hranách.  
Ukázka se nalézá níže. *TaskNode* tu používáme jen pro formu (nicméně výstup je vidět aspoň v logu). U hrany pro "buniness" si všimněme, že při pokračování k dalšímu nodu nepoužijeme parametr *score*, nýbrž *then*.  
A ano, králící popcorning nedělají, tj. asi jsme opět narazili na to, že GPT-4.1 mini není pro takového vyhodnocování zrovna nejlepší model :D.

In [15]:
from deepeval.metrics.dag import TaskNode, NonBinaryJudgementNode

name_extraction = TaskNode(
    instructions="Extract name of the pet.",
    output_label="Pet's name",
    evaluation_params=[SingleTurnParams.INPUT],
) 

buniness = BinaryJudgementNode(
    criteria="Is the text talking about bunnies?",
    evaluation_params=[
        SingleTurnParams.INPUT,
    ],
)

correctness = NonBinaryJudgementNode(
    criteria="Examine if the answer to the question is factually right.",
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT],
)

name_extraction.add_node(buniness)

buniness.add_verdict(verdict=True, then=correctness)
buniness.add_verdict(verdict=False, score=0)

correctness.add_verdict(verdict="Answer is correct", score=10)
correctness.add_verdict(verdict="Answer is partially correct", score=5)
correctness.add_verdict(verdict="Answer is incorrect", score=0)

dag_metric = DAGMetric(
    model="gpt-4.1-mini",
    name="custom_dag_metric",
    dag=DeepAcyclicGraph(root_nodes=[name_extraction]),
    verbose_mode=True
)

test_case = LLMTestCase(
    input="Do rabbits - or at least my bunny Bobek Ušatý - perform popcorning?",
    actual_output="Yes, they do.",
)

eval_object = evaluate(test_cases=[test_case], metrics=[dag_metric])

✨ You're running DeepEval's latest custom_dag_metric [DAG] Metric! (using gpt-4.1-mini, strict=False, 
async_mode=True)...

**************************************************

custom_dag_metric [DAG] Verbose Logs

**************************************************

______________________
| TaskNode | Level == 0 |
*******************************
Label: None

Instructions:
Extract name of the pet.

Pet's name:
Bobek Ušatý
 
 
__________________________________
| BinaryJudgementNode | Level == 1 |
************************************************
Label: None

Criteria:
Is the text talking about bunnies?

Verdict: True
Reason: The text explicitly mentions 'my bunny Bobek Ušatý' and asks about rabbits performing popcorning, 
indicating that it is talking about bunnies.
 
 
_____________________________________
| NonBinaryJudgementNode | Level == 2 |
*****************************************************
Label: None

Criteria:
Examine if the answer to the question is factually right.

Verdict: Answer is correct
Reason: Popcorning is a well-known behavior in rabbits where they jump and twist in the air, often as a sign of 
happiness or excitement. Therefore, stating that rabbits, including the user's bunny Bobek Ušatý, perform 
popcorning is factually accurate.
 
 
________________________
| VerdictNode | Level == 3 |
**********************************
Verdict: Answer is correct
Type: Deterministic
 
Score: 1.0
Reason: The score is 1.0 because the DAG traversal confirms that the text is about bunnies (BinaryJudgementNode 
verdict: True), and the answer provided is factually correct regarding rabbit behavior (NonBinaryJudgementNode 
verdict: Answer is correct), leading to a final VerdictNode confirming correctness.

======================================================================

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                            ┃ Average Score       ┃ Pass Rate                                 ┃ Total    │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━ │
│  custom_dag_metric [DAG]           │ 1.00                │ 100.00% | passed=1 | failed=0             │ 1        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=5422798;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 9.83s | token cost: 0.0007692 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

## Answer relevancy
Nyní se podívejme na několik metrik, které se týkajíc vyhodnocování RAGů. První z nich je answer relevancy. Ten říká, jak relevantní je actual_output v kontextu ke změní inputu. Funguje to tak, že LLMko napřed z actual_outputu vydoluje všechna tvrzení a posléze je jedno po druhém probere a vyhodnotí, nakolik jsou pro input důležité. Výsledná relevance se pak spočítá podle vzorce

$$ \text{answer relevancy} = \frac{\text{počet relevantních tvrzení}}{\text{počet tvrzení}} $$

Vidíme, že v příkladu to skončilo podle očekávání - ve výstupu byla tři tvrzení, z nichž jen jedno bylo pro otázku relevantní.

In [16]:
from deepeval.metrics import AnswerRelevancyMetric

answer_relevancy_metric = AnswerRelevancyMetric(
    model="gpt-4.1-mini",
    verbose_mode=True
)

test_case = LLMTestCase(
    input="Jak se říká králíčím skokům, které dělají, když jsou šťastní?",
    actual_output="Králíci - jedni z nejoblíbenějších mazlíčků - dělají skoky zvané binky. Krom jiného mají rádi seno", 
)

eval_object = evaluate([test_case], [answer_relevancy_metric])

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4.1-mini, strict=False, async_mode=True)...

**************************************************

Answer Relevancy Verbose Logs

**************************************************

Statements:
[
    "Králíci jsou jedni z nejoblíbenějších mazlíčků.",
    "Králíci dělají skoky zvané binky.",
    "Králíci mají rádi seno."
] 
 
Verdicts:
[
    {
        "verdict": "no",
        "reason": "This statement talks about rabbits being popular pets, which is not relevant to the name of the 
happy jumps they do."
    },
    {
        "verdict": "yes",
        "reason": null
    },
    {
        "verdict": "no",
        "reason": "This statement mentions rabbits liking hay, which does not address the question about the name 
of their happy jumps."
    }
]
 
Score: 0.3333333333333333
Reason: The score is 0.33 because the response includes irrelevant information about rabbits being popular pets and
their liking for hay, which does not answer the question about the name of the happy jumps they do. However, it is 
not lower because it still addresses rabbits and their behavior to some extent.

======================================================================

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_0                                                                                                 │
│  ├──   Input:            Jak se říká králíčím skokům, které dělají, když jsou šťastní?                          │
│  │     Actual Output:    Králíci - jedni z nejoblíbenějších mazlíčků - dělají skoky zvané binky. Krom jiného    │
│  │                       mají rádi seno                                                                         │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric           ┃ Score ┃ Threshold ┃ Reason                                                    │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        FAIL  │ Answer Relevancy │ 0.33  │ 0.50      │ The score is 0.33 because the response includes           │
│              │                  │       │           │ irrelevant information about rabbits being popular pets   │
│              │                  │       │           │ and their liking for hay, which does not answer the       │
│              │                  │       │           │ question about the name of the happy jumps they do.       │
│              │                  │       │           │ However, it is not lower because it still addresses       │
│              │                  │       │           │ rabbits and their behavior to some extent.                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                      ┃ Average Score          ┃ Pass Rate                                   ┃ Total     │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━ │
│  Answer Relevancy            │ 0.33                   │ 0.00% | passed=0 | failed=1                 │ 1         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=5422800;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 7.33s | token cost: 0.0008376 USD)
» Test Results (1 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

## Faithfulness
Faithfulness řeší, zda to, co se nalézá v actual_output, opravdu pochází z fragmentů textů získaných v rámci retrieval fáze RAGu (tj. z retrieval_context). Při evaluaci se zde nejprve z actual_outputu vypreparují všechna tvrzení. Pak se probírají jednotlivá tvrzení a zkoumá se, zda mají oporu ve fragmentech z kontextu.  
$$ \text{faithfulness} = \frac{\text{počet ozdrojovaných tvrzení}}{\text{počet tvrzení}} $$
Pozn.: z příkladu níže to téměř vypadá, jako by šlo jen o odhalení tvrzení odporujících kontextu.

In [20]:
from deepeval.metrics import FaithfulnessMetric

faitfullness_metric = FaithfulnessMetric(
    model="gpt-4.1-mini",
    verbose_mode=True
)

test_case = LLMTestCase(
    input="Jak se říká králíčím skokům, které dělají, když jsou šťastní?",
    actual_output="Králíci - jedni z nejoblíbenějších mazlíčků - dělají skoky zvané binky. Krom jiného mají rádi seno", 
    retrieval_context=[
        "Králíci jsou společně s psi, kočkami a křečky nejoblíbenější mazlíčci",
        "Binky jsou králičí skoky singalizující velkou spokojenost",
        "Bugs Bunny rád chroustá mrkev",
        "Kryštof Kolumbus objevil Ameriku"
    ]
)

eval_object = evaluate([test_case], [faitfullness_metric])

✨ You're running DeepEval's latest Faithfulness Metric! (using gpt-4.1-mini, strict=False, async_mode=True)...

**************************************************

Faithfulness Verbose Logs

**************************************************

Truths (limit=None):
[
    "Králíci jsou společně s psy, kočkami a křečky nejoblíbenější mazlíčci.",
    "Binky jsou králičí skoky signalizující velkou spokojenost.",
    "Bugs Bunny rád chroustá mrkev.",
    "Kryštof Kolumbus objevil Ameriku."
] 
 
Claims:
[
    "Králíci jsou jedni z nejoblíbenějších mazlíčků.",
    "Králíci dělají skoky zvané binky.",
    "Králíci mají rádi seno."
] 
 
Verdicts:
[
    {
        "verdict": "yes",
        "reason": null
    },
    {
        "verdict": "yes",
        "reason": null
    },
    {
        "verdict": "idk",
        "reason": "The context does not mention whether rabbits like hay, so this claim cannot be confirmed or 
contradicted."
    }
]
 
Score: 1.0
Reason: The score is 1.00 because there are no contradictions; the actual output fully aligns with the retrieval 
context. Great job maintaining accuracy!

======================================================================

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                ┃ Average Score           ┃ Pass Rate                                       ┃ Total      │
│ ━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━ │
│  Faithfulness          │ 1.00                    │ 100.00% | passed=1 | failed=0                   │ 1          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=5422808;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 5.02s | token cost: 0.0009920000000000003 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

## Contextual precision
Narozdíl od dvou předešlých RAGových metrik se contextual precision nekouká na output RAGu. Namísto toho řeší, zda (v kontextu inputu) jsou relevantní fragmenty textů přišlých z retrieveru výše než ty nerelevantní. Tj. fakt tu nejde jen o přítomnost relevantích fragmentů, ale o jejich pozici. V jazyce rovnic to znamená
$$ \text{Contextual precision} = \frac{1}{\text{Celkový počet relevantních fragmentů}}  \sum_{k=1}^{n} \frac{\text{Počet relevatních fragmentů až do pozice k}}{k}$$

Z hlediska parametrů v LLMTestCase nepotřebujeme *actual_output*, nicméně *expected_output* je povinný.

In [4]:
from deepeval.metrics import ContextualPrecisionMetric

con_precision_metric = ContextualPrecisionMetric(
    model="gpt-4.1-mini",
    verbose_mode=True
)

test_case = LLMTestCase(
    input="Jak se říká králíčím skokům, které dělají, když jsou šťastní?",
    expected_output="Těmto skokům se říká 'binky'",
    #actual_output="Králíci - jedni z nejoblíbenějších mazlíčků - dělají skoky zvané binky. Krom jiného mají rádi seno", 
    retrieval_context=[
        "Králíci jsou společně s psi, kočkami a křečky nejoblíbenější mazlíčci",
        "Binky jsou králičí skoky singalizující velkou spokojenost",
        "Bugs Bunny rád chroustá mrkev",
        "Kryštof Kolumbus objevil Ameriku"
    ]
)

eval_object = evaluate([test_case], [con_precision_metric])

✨ You're running DeepEval's latest Contextual Precision Metric! (using gpt-4.1-mini, strict=False, 
async_mode=True)...

**************************************************

Contextual Precision Verbose Logs

**************************************************

Verdicts:
[
    {
        "verdict": "no",
        "reason": "The statement 'Kr\u00e1l\u00edci jsou spole\u010dn\u011b s psi, ko\u010dkami a k\u0159e\u010dky 
nejobl\u00edben\u011bj\u0161\u00ed mazl\u00ed\u010dci' talks about rabbits being popular pets but does not mention 
anything about their happy jumps."
    },
    {
        "verdict": "yes",
        "reason": "The text 'Binky jsou kr\u00e1li\u010d\u00ed skoky singalizuj\u00edc\u00ed velkou spokojenost' 
directly explains that 'Binky' are rabbit jumps signaling great happiness, which answers the question about what 
these happy jumps are called."
    },
    {
        "verdict": "no",
        "reason": "'Bugs Bunny r\u00e1d chroust\u00e1 mrkev' is about a cartoon rabbit eating carrots and does not 
relate to the name of happy rabbit jumps."
    },
    {
        "verdict": "no",
        "reason": "'Kry\u0161tof Kolumbus objevil Ameriku' is unrelated to rabbits or their behavior."
    }
]
 
Score: 0.5
Reason: The score is 0.50 because the first node in the retrieval contexts, ranked 1st, is irrelevant as it 
discusses rabbits as popular pets without mentioning their happy jumps, while the relevant node explaining 'Binky' 
as the term for happy rabbit jumps is ranked 2nd. This misplacement of an irrelevant node above a relevant one 
lowers the score. However, the relevant node is still ranked higher than the other irrelevant nodes at ranks 3 and 
4, which justifies the score not being lower.

======================================================================

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                         ┃ Average Score        ┃ Pass Rate                                   ┃ Total    │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━ │
│  Contextual Precision           │ 0.50                 │ 100.00% | passed=1 | failed=0               │ 1        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=1895695;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 5.57s | token cost: 0.0009608 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

## Contextual recall
Contextual recall zkoumá výstup retrievalu. Tato metrika vezme expected_output a s pomocí LLMka z něj vypreparuje všechna tvrzení. Poté se dívá, zda tato tvrzení mají oporu ve fragmentech z retrievalu. Vzorec vypadá stejně jako ve faithfulness - liší se ale tím, že ozdrojovaná tvrzení se tentokrát vztahují k očekávanému a ne skutečnému výstupu.  
$$ \text{Contextual recall} = \frac{\text{počet ozdrojovaných tvrzení}}{\text{počet tvrzení}} $$

In [5]:
from deepeval.metrics import ContextualRecallMetric

con_recall_metric = ContextualRecallMetric(
    model="gpt-4.1-mini",
    verbose_mode=True
)

test_case = LLMTestCase(
    input="Jak se říká králíčím skokům, které dělají, když jsou šťastní?",
    expected_output="Těmto skokům se říká 'binky'",
    retrieval_context=[
        "Králíci jsou společně s psi, kočkami a křečky nejoblíbenější mazlíčci",
        "Binky jsou králičí skoky singalizující velkou spokojenost",
        "Bugs Bunny rád chroustá mrkev",
        "Kryštof Kolumbus objevil Ameriku"
    ]
)

eval_object = evaluate([test_case], [con_recall_metric])

✨ You're running DeepEval's latest Contextual Recall Metric! (using gpt-4.1-mini, strict=False, 
async_mode=True)...

**************************************************

Contextual Recall Verbose Logs

**************************************************

Verdicts:
[
    {
        "verdict": "yes",
        "reason": "2nd node: 'Binky jsou kr\u00e1li\u010d\u00ed skoky...' relates to 'T\u011bmto skok\u016fm se 
\u0159\u00edk\u00e1 'binky'.",
        "expected_output": "T\u011bmto skok\u016fm se \u0159\u00edk\u00e1 'binky'"
    }
]
 
Score: 1.0
Reason: The score is 1.00 because the sentence in the expected output directly corresponds to information in node 2
of the retrieval context, which defines 'binky' as rabbit jumps, perfectly supporting the statement without any 
contradictions.

======================================================================

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                      ┃ Average Score         ┃ Pass Rate                                    ┃ Total     │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━ │
│  Contextual Recall           │ 1.00                  │ 100.00% | passed=1 | failed=0                │ 1         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=1895697;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 4.4s | token cost: 0.000518 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

## Contextual relevancy
I tato metrika sleduje výstup retrieveru - tentokrát ale prismatem inputu. Tj. vezmou se všechna tvrzení z retriever contextu a pak se zkoumá, zda jsou relevantní vzhledem k inputu.

$$ \text{Contextual relevancy} = \frac{\text{počet relevantních tvrzení}}{\text{počet tvrzení}} $$


In [6]:
from deepeval.metrics import ContextualRelevancyMetric

con_relevancy_metric = ContextualRelevancyMetric(
    model="gpt-4.1-mini",
    verbose_mode=True
)

test_case = LLMTestCase(
    input="Jak se říká králíčím skokům, které dělají, když jsou šťastní?",
    retrieval_context=[
        "Králíci jsou společně s psi, kočkami a křečky nejoblíbenější mazlíčci",
        "Binky jsou králičí skoky singalizující velkou spokojenost",
        "Bugs Bunny rád chroustá mrkev",
        "Kryštof Kolumbus objevil Ameriku"
    ]
)

eval_object = evaluate([test_case], [con_relevancy_metric])

✨ You're running DeepEval's latest Contextual Relevancy Metric! (using gpt-4.1-mini, strict=False, 
async_mode=True)...

**************************************************

Contextual Relevancy Verbose Logs

**************************************************

Verdicts:
[
    {
        "verdicts": [
            {
                "statement": "Kr\u00e1l\u00edci jsou spole\u010dn\u011b s psi, ko\u010dkami a k\u0159e\u010dky 
nejobl\u00edben\u011bj\u0161\u00ed mazl\u00ed\u010dci",
                "verdict": "no",
                "reason": "The statement 'Kr\u00e1l\u00edci jsou spole\u010dn\u011b s psi, ko\u010dkami a 
k\u0159e\u010dky nejobl\u00edben\u011bj\u0161\u00ed mazl\u00ed\u010dci' does not address the term for the happy 
jumps that rabbits make."
            }
        ]
    },
    {
        "verdicts": [
            {
                "statement": "Binky jsou kr\u00e1li\u010d\u00ed skoky singalizuj\u00edc\u00ed velkou spokojenost",
                "verdict": "yes",
                "reason": null
            }
        ]
    },
    {
        "verdicts": [
            {
                "statement": "Bugs Bunny r\u00e1d chroust\u00e1 mrkev",
                "verdict": "no",
                "reason": "The statement 'Bugs Bunny r\u00e1d chroust\u00e1 mrkev' is about Bugs Bunny eating a 
carrot and does not relate to 'kr\u00e1l\u00ed\u010d\u00ed skoky' or rabbit jumps when they are happy."
            }
        ]
    },
    {
        "verdicts": [
            {
                "statement": "Kry\u0161tof Kolumbus objevil Ameriku",
                "verdict": "no",
                "reason": "The statement 'Kry\u0161tof Kolumbus objevil Ameriku' is about Christopher Columbus 
discovering America, which is unrelated to 'kr\u00e1l\u00ed\u010d\u00ed skoky' (rabbit jumps) and their meaning 
when rabbits are happy."
            }
        ]
    }
]
 
Score: 0.25
Reason: The score is 0.25 because only one statement, 'Binky jsou králičí skoky singalizující velkou spokojenost,' 
directly answers the question about happy rabbit jumps, while the other statements about popular pets, Bugs Bunny, 
and Columbus are irrelevant.

======================================================================

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_0                                                                                                 │
│  ├──   Input:            Jak se říká králíčím skokům, které dělají, když jsou šťastní?                          │
│  │     Actual Output:    None                                                                                   │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric               ┃ Score ┃ Threshold ┃ Reason                                                │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        FAIL  │ Contextual Relevancy │ 0.25  │ 0.50      │ The score is 0.25 because only one statement,         │
│              │                      │       │           │ 'Binky jsou králičí skoky singalizující velkou        │
│              │                      │       │           │ spokojenost,' directly answers the question about     │
│              │                      │       │           │ happy rabbit jumps, while the other statements        │
│              │                      │       │           │ about popular pets, Bugs Bunny, and Columbus are      │
│              │                      │       │           │ irrelevant.                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                          ┃ Average Score         ┃ Pass Rate                                ┃ Total     │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━ │
│  Contextual Relevancy            │ 0.25                  │ 0.00% | passed=0 | failed=1              │ 1         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=1895699;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 4.53s | token cost: 0.0015980000000000002 USD)
» Test Results (1 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

## Evaluace v rámci příkazové řádky
Výše jsme si ukázali vyhodnocování testů v rámci normálního kódu v Jupyter notebooku. Co ale dělat, když bychom chtěli deepeval scénáře použít jako svého druhu unit testy. Pomiňme nyní skutečnost, že by to (z hlediska časového i finančního) nemusel být zrovna nejlepší nápad.  
Skript i testovací funkce by měly mít prefix test. Pro vyhodnocování není nepoužijeme funkci *evaluate*, nýbrž *assert_test* - v té navíc nebude "test_case" v listu, nýbrž izolovaně.
```python
# obsah souboru test_deepeval_example.py
from deepeval import evaluate
from deepeval.test_case import LLMTestCase, SingleTurnParams
from deepeval.metrics import GEval
from deepeval import assert_test

def test_example():
    correctness_metric = GEval(
        model="gpt-4.1-mini",
        name="Correctness",
        criteria="Determine if the 'actual output' is correct based on the 'expected output'.",
        evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.EXPECTED_OUTPUT],
        threshold=0.5,
        verbose_mode=True
    )

    test_case = LLMTestCase(
        input="Když jsou králící šťastní, skáčou radostí?",
        actual_output="Ano, když jsou králící šťastní, tak skáčou.",
        expected_output="Ano, králíci skáčou radostí a tento projev je jasným znakem maximálního štěstí, bezpečí a životní energie. Tomuto specifickému chování se v chovatelské komunitě mezinárodně říká binky."
    )

    assert_test(test_case, [correctness_metric])
```
Pakliže máme v našem rootu i soubor ".env" (kvůli API klíči), tak by nyní mělo stačit 
```
deepeval test run test_deepeval_example.py
```